# Breast Cancer Molecular Subtype Classification from Whole-Slide Images
## Approach 3 — UNI features from 224 × 224 crops

**UNI** features (1024-D) extracted from 224 × 224 px crops of individual patches, selected by a quality filter (brightness and RGB standard deviation) and aggregated by mean pooling.

**Patch representation.** Identical to Approach 2 — one crop per 1000 × 1000 px patch, chosen as the best of five random 224 × 224 candidates by RGB standard deviation, with no resizing — so that the only difference between the two approaches is the feature extractor: UNI, a foundation model for computational pathology, instead of KimiaNet.

With the resulting matrix, a panel of classifiers (KNN, RF, LR, SVM, XGBoost) plus an MLP is trained on the transcriptomic train/test split, and the models are then applied to the external INC Bogotá cohort.

- **Inputs:** `UNI_patches_meanpool_552.csv`, `split_transcriptomica_604.csv` and `UNI_patches_meanpool_INC23.csv`.
- **Outputs:** summary table, confusion matrices, repeated cross-validation report and external-validation table.

## 1. Configuration and imports

### 1.1 Imports

Every library used in the notebook is imported once here. The classifiers, the cross-validation schemes and the metrics come from scikit-learn; XGBoost is imported separately.

In [ ]:
# ── Imports (consolidated) ───────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import randint, uniform, wilcoxon
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV, RandomizedSearchCV, cross_validate)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

In [ ]:
# ── Additional imports for the MLP ───────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

### 1.2 Environment record

Versions of the main libraries, recorded for reproducibility.

In [ ]:
%pip list --format=freeze > requirements_wsi.txt
!grep -iE "^(numpy|scipy|scikit-learn|xgboost|torch|timm|opencv|openslide)" requirements_wsi.txt

## 2. Data loading

### 2.1 Feature table and partition

The feature table (one row per patient) is merged with the train/test partition exported by the transcriptomic notebook, using the patient identifier as key. Reusing that partition is what keeps the two modalities paired: the same patients are in the training set here and there, which is required for the later multi-omic integration.

`np.nan_to_num` replaces any missing or infinite value with zero, because the distance-based models (KNN, SVM) cannot operate on NaN. Labels are mapped to integers with the same dictionary used across the project (LumA = 0, LumB = 1, Basal = 2).

In [ ]:
# ============================================================
# Data loading — UNI features (crops, WSI)
# ============================================================
RUTA_FEAT  = "UNI_patches_meanpool_552.csv"   # adjust to the location of the feature table
RUTA_SPLIT = "split_transcriptomica_604.csv"      # partition exported by the transcriptomic notebook
ENFOQUE    = "UNI_Crop"

MAP    = {"LumA": 0, "LumB": 1, "Basal": 2}
CLASES = ["LumA", "LumB", "Basal"]                       # order 0, 1, 2

feat_df  = pd.read_csv(RUTA_FEAT)
split_df = pd.read_csv(RUTA_SPLIT)
df_wsi   = feat_df.merge(split_df, left_on="ID", right_on="patient_id")

feat_cols  = [str(i) for i in range(1024)]
mask_train = df_wsi["split"] == "train"
mask_test  = df_wsi["split"] == "test"

X_train_wsi = np.nan_to_num(df_wsi.loc[mask_train, feat_cols].values.astype(float))
X_test_wsi  = np.nan_to_num(df_wsi.loc[mask_test,  feat_cols].values.astype(float))
y_train_wsi = df_wsi.loc[mask_train, "Subtype"].map(MAP).values   # 0/1/2 from the start
y_test_wsi  = df_wsi.loc[mask_test,  "Subtype"].map(MAP).values

print(f"Train: {X_train_wsi.shape} | Test: {X_test_wsi.shape}")
print("Class distribution (train):", np.bincount(y_train_wsi))

### 2.2 Cross-validation schemes

Two schemes are defined and kept separate: `cv` for hyperparameter search inside the training set, and `cv_report` (5 folds × 3 repeats) for the final comparison of the tuned models. `scoring` collects the macro-averaged metrics, which weight the three subtypes equally.

In [ ]:
cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)          # tuning
cv_report = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)   # final report
scoring   = {"F1_macro": "f1_macro", "Balanced_acc": "balanced_accuracy",
             "Precision": "precision_macro", "Recall": "recall_macro"}

## 3. Classical classifiers

Five classifiers are tuned on the training set by stratified cross-validation, then evaluated on the held-out test set. The search space matches the one used in the transcriptomic phase, so that the two modalities are compared under the same protocol.

For the models that are sensitive to feature scale (logistic regression, SVM) the scaler sits inside the pipeline, so it is re-fitted within each fold and no information leaks from the validation folds. Random forest and XGBoost do not need it, being tree-based.

### 3.1 Model 1: k-nearest neighbours

Grid search over the number of neighbours, the weighting scheme, the metric and the Minkowski exponent (360 configurations × 5 folds).

In [ ]:
# =============================================
# MODEL 1: KNN
# =============================================
print("Training KNN...")
knn_wsi = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': range(1, 31),
    'weights':     ['uniform', 'distance'],
    'metric':      ['minkowski', 'cosine'],
    'p':           [1, 2, 3]
}

grid_knn_wsi = GridSearchCV(
    knn_wsi,
    param_grid=param_grid_knn,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_knn_wsi.fit(X_train_wsi, y_train_wsi)

best_knn_wsi   = grid_knn_wsi.best_estimator_
y_pred_knn_wsi = best_knn_wsi.predict(X_test_wsi)

print("Best KNN hyperparameters:", grid_knn_wsi.best_params_)
print(f"Train accuracy: {best_knn_wsi.score(X_train_wsi, y_train_wsi):.4f}")
print(f"Test accuracy:  {accuracy_score(y_test_wsi, y_pred_knn_wsi):.4f}")
print(classification_report(y_test_wsi, y_pred_knn_wsi, target_names=CLASES))

### 3.2 Model 2: Random forest

Randomised search (50 sampled configurations) over the number and depth of the trees, the splitting criterion and the fraction of features considered at each split.

In [ ]:
# =============================================
# MODEL 2: Random Forest
# =============================================
print("Training Random Forest...")
rf_wsi = RandomForestClassifier(random_state=42, n_jobs=-1)

param_dist_rf = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [None, 30, 40, 50],
    "min_samples_split":[2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features":     [0.02, 0.05, 0.1, 0.15, 0.18],
    "criterion":        ["gini", "entropy"]
}

rs_rf_wsi = RandomizedSearchCV(
    rf_wsi,
    param_distributions=param_dist_rf,
    n_iter=50,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
rs_rf_wsi.fit(X_train_wsi, y_train_wsi)

best_rf_wsi      = rs_rf_wsi.best_estimator_
y_pred_rf_wsi    = best_rf_wsi.predict(X_test_wsi)
y_train_pred_rf  = best_rf_wsi.predict(X_train_wsi)

print("Best RF hyperparameters:", rs_rf_wsi.best_params_)
print(f"Train accuracy: {accuracy_score(y_train_wsi, y_train_pred_rf):.4f}")
print(f"Test accuracy:  {accuracy_score(y_test_wsi, y_pred_rf_wsi):.4f}")
print(classification_report(y_test_wsi, y_pred_rf_wsi, target_names=CLASES))

### 3.3 Model 3: Logistic regression

A linear baseline. The grid is written as a list of dictionaries because scikit-learn does not accept every penalty / solver / multi-class combination: `liblinear` is one-vs-rest only, `lbfgs` and `newton-cg` take only the L2 penalty, and `saga` takes both.

In [ ]:
# =============================================
# MODEL 3: Logistic Regression
# =============================================
print("Training Logistic Regression...")
lr_pipeline_wsi = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42, max_iter=1000))
])

# Written as a list of dictionaries: not every penalty / solver / multi_class
# combination is valid in scikit-learn.
param_grid_lr = [
    {
        'lr__C':           [0.001, 0.01, 0.1, 1],
        'lr__penalty':     ['l1', 'l2'],
        'lr__solver':      ['liblinear'],
        'lr__multi_class': ['ovr']
    },
    {
        'lr__C':           [0.001, 0.01, 0.1, 1],
        'lr__penalty':     ['l2'],
        'lr__solver':      ['lbfgs', 'newton-cg'],
        'lr__multi_class': ['ovr', 'multinomial']
    },
    {
        'lr__C':           [0.001, 0.01, 0.1, 1],
        'lr__penalty':     ['l1', 'l2'],
        'lr__solver':      ['saga'],
        'lr__multi_class': ['ovr', 'multinomial']
    }
]

grid_lr_wsi = GridSearchCV(
    lr_pipeline_wsi,
    param_grid=param_grid_lr,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_lr_wsi.fit(X_train_wsi, y_train_wsi)

best_lr_wsi    = grid_lr_wsi.best_estimator_
y_pred_lr_wsi  = best_lr_wsi.predict(X_test_wsi)

print("Best LR hyperparameters:", grid_lr_wsi.best_params_)
print(f"Best CV accuracy: {grid_lr_wsi.best_score_:.4f}")
print(f"Test accuracy:    {accuracy_score(y_test_wsi, y_pred_lr_wsi):.4f}")
print(classification_report(y_test_wsi, y_pred_lr_wsi, target_names=CLASES))

### 3.4 Model 4: Support vector machine

Polynomial and RBF kernels, preceded by standardisation. `probability=True` fits the Platt calibration needed for probability estimates, at the cost of a slower fit. `degree` and `coef0` affect only the polynomial kernel.

In [ ]:
# =============================================
# MODEL 4: SVM
# =============================================
print("Training SVM...")
svm_pipeline_wsi = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42, probability=True))
])

param_grid_svm = {
    'svc__C':      [0.01, 0.1, 1, 10, 100],
    'svc__kernel': ['poly', 'rbf'],
    'svc__gamma':  ['scale', 1, 0.1, 0.01, 0.001, 0.0001],
    'svc__degree': [2, 3, 4],        # polynomial kernel only
    'svc__coef0':  [0.0, 0.1, 1.0]   # polynomial kernel only
}

grid_svm_wsi = GridSearchCV(
    svm_pipeline_wsi,
    param_grid=param_grid_svm,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_svm_wsi.fit(X_train_wsi, y_train_wsi)

best_svm_wsi   = grid_svm_wsi.best_estimator_
y_pred_svm_wsi = best_svm_wsi.predict(X_test_wsi)

print("Best SVM hyperparameters:", grid_svm_wsi.best_params_)
print(f"Train accuracy: {best_svm_wsi.score(X_train_wsi, y_train_wsi):.4f}")
print(f"Test accuracy:  {accuracy_score(y_test_wsi, y_pred_svm_wsi):.4f}")
print(classification_report(y_test_wsi, y_pred_svm_wsi, target_names=CLASES))

### 3.5 Model 5: XGBoost

Gradient-boosted trees, preceded by univariate feature selection (`SelectKBest`). Class-balanced sample weights are passed to the fit so that the smallest subtype is not neglected. Sixty configurations are sampled; `uniform(loc, scale)` draws from [`loc`, `loc + scale`].

In [ ]:
# =============================================
# MODEL 5: XGBoost
# =============================================
print("Training XGBoost...")
# The labels are already 0/1/2; LabelEncoder is kept for consistency with the
# original pipeline (here it acts as an identity transform).
le_wsi = LabelEncoder()
y_train_wsi_enc = le_wsi.fit_transform(y_train_wsi)
y_test_wsi_enc  = le_wsi.transform(y_test_wsi)

# Class weights, inversely proportional to class frequency
cls_counts_wsi   = pd.Series(y_train_wsi_enc).value_counts()
class_weight_wsi = (cls_counts_wsi.sum() /
                    (len(cls_counts_wsi) * cls_counts_wsi)).to_dict()
sample_weight_wsi = pd.Series(y_train_wsi_enc).map(class_weight_wsi).values

pipe_xgb_wsi = Pipeline(steps=[
    ("sel", SelectKBest(score_func=f_classif)),
    ("clf", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        tree_method="hist",
        n_jobs=-1,
        eval_metric="mlogloss",
        random_state=42
    ))
])

param_dist_xgb = {
    "sel__k":                [100, 200, 400, "all"],
    "clf__n_estimators":     randint(200, 600),
    "clf__max_depth":        randint(3, 8),
    "clf__learning_rate":    uniform(0.05, 0.3),
    "clf__subsample":        uniform(0.6, 0.3),
    "clf__colsample_bytree": uniform(0.4, 0.4),
    "clf__min_child_weight": randint(1, 5),
    "clf__reg_lambda":       uniform(0.5, 2.0),
    "clf__reg_alpha":        uniform(0.0, 1.0),
}

rs_xgb_wsi = RandomizedSearchCV(
    pipe_xgb_wsi,
    param_distributions=param_dist_xgb,
    n_iter=60,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rs_xgb_wsi.fit(X_train_wsi, y_train_wsi_enc,
               clf__sample_weight=sample_weight_wsi)

best_xgb_wsi        = rs_xgb_wsi.best_estimator_
y_pred_xgb_wsi_enc  = best_xgb_wsi.predict(X_test_wsi)
y_pred_xgb_wsi      = le_wsi.inverse_transform(y_pred_xgb_wsi_enc)

print("Best XGB hyperparameters:", rs_xgb_wsi.best_params_)
print(f"Test accuracy: {accuracy_score(y_test_wsi, y_pred_xgb_wsi_enc):.4f}")
print(classification_report(y_test_wsi_enc, y_pred_xgb_wsi_enc, target_names=CLASES))

### 3.6 Summary on the test set

The five models side by side on the held-out samples. Macro-averaged precision, recall and F1 give every subtype the same weight regardless of its size.

In [ ]:
# =============================================
# SUMMARY TABLE
# =============================================
modelos_wsi = {
    'KNN':                 y_pred_knn_wsi,
    'Random Forest':       y_pred_rf_wsi,
    'Logistic Regression': y_pred_lr_wsi,
    'SVM':                 y_pred_svm_wsi,
    'XGBoost':             y_pred_xgb_wsi,   # decoded predictions, not the encoded ones
}

resultados_wsi = []
for nombre, preds in modelos_wsi.items():
    resultados_wsi.append({
        'Model':     nombre,
        'Accuracy':  round(accuracy_score(y_test_wsi, preds), 3),
        'Precision': round(precision_score(y_test_wsi, preds, average='macro', zero_division=0), 3),
        'Recall':    round(recall_score(y_test_wsi, preds, average='macro', zero_division=0), 3),
        'F1 macro':  round(f1_score(y_test_wsi, preds, average='macro', zero_division=0), 3),
    })

df_resultados_wsi = pd.DataFrame(resultados_wsi)
print(f"======= SUMMARY — {ENFOQUE} (test set) =======")
print(df_resultados_wsi.to_string(index=False))

In [ ]:
# Dictionary of the five tuned models (used by the cross-validation report)
modelos = {"KNN": best_knn_wsi, "Random Forest": best_rf_wsi,
           "Logistic Regression": best_lr_wsi,
           "XGBoost": best_xgb_wsi, "SVM": best_svm_wsi}

## 4. Evaluation by repeated cross-validation

A single test set gives a noisy estimate, so the tuned models are re-evaluated on the whole cohort with 5-fold stratified cross-validation repeated three times (15 folds per model). Because every model sees the same folds, the per-fold scores are paired and the two best-ranked models can be compared with a Wilcoxon signed-rank test, which asks whether their difference is systematic or within fold-to-fold noise.

In [ ]:
def reporte_cv(modelos, X_train_wsi, X_test_wsi, y_train_wsi, y_test_wsi):
    """Repeated stratified cross-validation of every model on the whole cohort.

    Returns the summary table (mean +/- SD per metric) and the per-fold F1-macro
    scores of each model, which are paired and feed the Wilcoxon test.
    """
    X = np.vstack([X_train_wsi, X_test_wsi])
    y = np.concatenate([y_train_wsi, y_test_wsi])
    filas, f1_folds = [], {}
    for nombre, mod in modelos.items():
        r = cross_validate(mod, X, y, cv=cv_report, scoring=scoring, n_jobs=-1)
        f1_folds[nombre] = r["test_F1_macro"]          # per-fold F1 (for Wilcoxon)
        fila = {"Model": nombre}
        for m in scoring:
            fila[m] = f"{r['test_'+m].mean():.3f} +/- {r['test_'+m].std():.3f}"
        filas.append(fila)
    tabla = pd.DataFrame(filas).set_index("Model").sort_values("F1_macro", ascending=False)
    return tabla, f1_folds

Consistency check of the combined matrix before running the cross-validation.

In [ ]:
X = np.vstack([X_train_wsi, X_test_wsi])
y = np.concatenate([y_train_wsi, y_test_wsi])

print("Xtr:", X_train_wsi.shape, "| Xte:", X_test_wsi.shape)
print("Combined X:", X.shape, "| y:", y.shape)
print("Row counts consistent?", X.shape[0] == X_train_wsi.shape[0] + X_test_wsi.shape[0] == len(y))
print("Class distribution:", np.bincount(y))

In [ ]:
tabla_cv, f1_folds = reporte_cv(modelos, X_train_wsi, X_test_wsi, y_train_wsi, y_test_wsi)
print(f"--- {ENFOQUE} · repeated CV ---")
print(tabla_cv.to_string())

# Wilcoxon signed-rank test between the two best-ranked models
top2 = tabla_cv.index[:2].tolist()
stat, p = wilcoxon(f1_folds[top2[0]], f1_folds[top2[1]])
print(f"\nWilcoxon {top2[0]} vs {top2[1]}: p = {p:.3f}  ->",
      "significant difference" if p < 0.05 else "equivalent performance")

## 5. Multilayer perceptron (MLP)

A small fully connected network is added as a sixth classifier, to test whether a non-linear model extracts more subtype signal from the embeddings than the classical models. The architecture is 512-256-128 with batch normalisation, ReLU and dropout, trained with AdamW, a cosine learning-rate schedule and class-weighted cross-entropy.

In [ ]:
# Multilayer perceptron used as a sixth classifier. It is evaluated on the fixed
# split (not by repeated CV) because of its training cost; this is stated in the
# methods. Labels follow the same convention as the rest of the pipeline:
# LumA = 0, LumB = 1, Basal = 2.

class MLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=(512, 256, 128), n_classes=3, dropout=0.4):
        super().__init__()
        capas, prev = [], input_dim
        for h in hidden_dims:
            capas += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        capas.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*capas)

    def forward(self, x):
        return self.net(x)


def entrenar_mlp(Xtr, Xte, ytr, yte, epochs=150, lr=1e-3, batch=32, seed=42):
    """Train the MLP and return (best_f1, y_pred_test, history).

    ytr/yte are already encoded 0/1/2; no LabelEncoder is applied.
    NOTE: the epoch kept is the one with the highest F1 on the TEST set, so the
    reported figure is optimistic (see the notes at the end of the notebook).
    """
    torch.manual_seed(seed)

    # Standardisation (fitted on train only, no leakage)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xte_s = sc.transform(Xte)

    X_tr = torch.FloatTensor(Xtr_s); y_tr = torch.LongTensor(ytr)
    X_te = torch.FloatTensor(Xte_s); y_te = torch.LongTensor(yte)
    dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch, shuffle=True)

    model = MLP(input_dim=Xtr.shape[1])
    # Class weights for the imbalance
    counts  = np.bincount(ytr, minlength=3)
    weights = torch.FloatTensor(len(counts) / (len(counts) * counts))
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1, best_state = 0.0, None
    history = {"train_loss": [], "val_f1": []}

    for ep in range(epochs):
        model.train(); tot = 0.0
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
            tot += loss.item()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            vp = model(X_te).argmax(1).numpy()
        f1 = f1_score(y_te.numpy(), vp, average="macro")
        history["train_loss"].append(tot / len(dl))
        history["val_f1"].append(f1)
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        y_pred = model(X_te).argmax(1).numpy()
    return best_f1, y_pred, history

Training, evaluation and diagnostic curves. The loss curve shows whether the network converged; the F1 curve shows whether it is still improving or has started to overfit.

In [ ]:
# ── MLP as a sixth classifier — UNI_Crop ───────────────────────────────────
# Trained on the fixed split (train/test); no repeated CV, for computational cost.
# Labels: LumA = 0, LumB = 1, Basal = 2 (consistent with the rest of the pipeline).

best_f1_mlp, y_pred_mlp, history_mlp = entrenar_mlp(
    X_train_wsi, X_test_wsi,
    y_train_wsi, y_test_wsi
)

print(f"MLP ({ENFOQUE}) — best test F1-macro: {best_f1_mlp:.3f}\n")
print(classification_report(y_test_wsi, y_pred_mlp,
                            labels=[0, 1, 2], target_names=CLASES))

# Training curves + confusion matrix
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(history_mlp["train_loss"], color="steelblue")
ax[0].set_title("Train loss"); ax[0].set_xlabel("Epoch")

ax[1].plot(history_mlp["val_f1"], color="orange")
ax[1].axhline(best_f1_mlp, color="red", ls="--",
              label=f"Best F1={best_f1_mlp:.3f}")
ax[1].set_title("Test F1-macro"); ax[1].set_xlabel("Epoch"); ax[1].legend()

cm = confusion_matrix(y_test_wsi, y_pred_mlp, labels=[0, 1, 2])
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(
    ax=ax[2], cmap="Blues", values_format="d", colorbar=False)
ax[2].set_title(f"MLP — {ENFOQUE}")
plt.tight_layout(); plt.show()

## 6. External validation — INC Bogotá cohort (n = 23, Luminal B)

The models tuned on TCGA-BRCA are applied, without any retraining or re-selection, to an independent cohort of 23 Luminal B cases. With a single class present, accuracy and macro-F1 are not meaningful; the interpretable metric is the **sensitivity for Luminal B**, i.e. the fraction of samples assigned to the correct class.

### 6.1 Sensitivity for Luminal B

In [ ]:
# ── External validation — INC cohort (n=23, Luminal B) — UNI_Crop ──────────
# Features extracted from the INC slides with the same pipeline used on
# TCGA-BRCA. The models are fixed: no retraining, no re-selection.

RUTA_INC = "UNI_patches_meanpool_INC23.csv"   # adjust

df_inc    = pd.read_csv(RUTA_INC)
X_inc_wsi = np.nan_to_num(df_inc[feat_cols].values.astype(float))
y_inc_wsi = np.full(len(df_inc), MAP["LumB"])     # all LumB by cohort design

filas = []
for nombre, mod in modelos.items():
    yp   = mod.predict(X_inc_wsi)
    n_ok = int((yp == MAP["LumB"]).sum())
    filas.append({
        "Model":            nombre,
        "Correct LumB":     n_ok,
        "Total":            len(yp),
        "LumB sensitivity": round(n_ok / len(yp), 3)
    })

tabla_inc = pd.DataFrame(filas).set_index("Model")
print(f"===== LumB sensitivity — INC (n={len(y_inc_wsi)}) — {ENFOQUE} =====")
print(tabla_inc.to_string())

### 6.2 Where do the predictions collapse?

A low sensitivity can mean two different things: the samples are spread across the three classes (weak signal), or they concentrate in a single wrong class (systematic shift). The distribution below distinguishes the two cases.

In [ ]:
# Distribution of the predictions: where do they collapse?
from collections import Counter

pred_texto = {0: "LumA", 1: "LumB", 2: "Basal"}
for nombre, mod in modelos.items():
    yp = mod.predict(X_inc_wsi)
    conteo = Counter([pred_texto[p] for p in yp])
    print(f"{nombre}: {dict(conteo)}")

### 6.3 Comparability of the embeddings

If the predictions collapse into one class, the most likely cause is that the INC embeddings do not live on the same scale as those the models were trained on. The mean, the standard deviation and the per-patient L2 norm of both cohorts are compared below. A systematic difference in the L2 norm indicates a domain shift in the feature extractor's output, not a biological difference between the cohorts.

In [ ]:
# Compare the embedding distributions: INC vs TCGA train
print("TCGA — mean:", X_train_wsi.mean().round(4),
      "| std:", X_train_wsi.std().round(4))
print("INC  — mean:", X_inc_wsi.mean().round(4),
      "| std:", X_inc_wsi.std().round(4))

# Per-patient L2 norm
norma_tcga = np.linalg.norm(X_train_wsi, axis=1)
norma_inc  = np.linalg.norm(X_inc_wsi,  axis=1)
print(f"L2 norm TCGA — mean: {norma_tcga.mean():.2f} | std: {norma_tcga.std():.2f}")
print(f"L2 norm INC  — mean: {norma_inc.mean():.2f}  | std: {norma_inc.std():.2f}")

Finally, a check that the scale-sensitive classifiers do carry their own internal scaler: if they do, the shift above is not corrected by re-scaling the input, since the scaler was fitted on TCGA.

In [ ]:
# Check whether the classifier carries an internal scaler
print(best_lr_wsi)
print(best_svm_wsi)

## Notes and limitations

1. **MLP model selection.** The epoch kept is the one with the highest F1 on the test set, so the reported F1 is an optimistic estimate rather than a clean held-out result. A separate validation split drawn from the training set would be needed to report it as such.
2. **External validation.** The INC embeddings were extracted with the same pipeline but on slides from a different centre, scanner and staining protocol. The comparison of the L2 norms quantifies that shift; because the classifiers carry a scaler fitted on TCGA, the shift is not corrected at prediction time, and the reported sensitivity should be read as a measure of robustness under domain shift rather than as clinical accuracy.
3. **Repeated cross-validation.** It is run on the whole cohort with hyperparameters tuned on the training partition, which slightly favours every model. Folds also share training data, so the per-fold scores are not independent and the Wilcoxon *p*-value is approximate.
4. **XGBoost class weights.** The sample weights used during tuning are not passed to `cross_validate`, so the repeated-CV section refits XGBoost without class weighting.
5. **Patient-level split.** The partition comes from the transcriptomic notebook; a patient present twice in the slide table (two slides for the same barcode) would appear in both the training and the test sets. It is worth checking for duplicated identifiers before merging.